# Chapter 07: Advanced Text Generation Techniques and tools

In [4]:
# %%capture
!pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community

# Fix: Use GGML_CUDA=on instead of LLAMA_CUDA=on
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.2.69 --force-reinstall --no-cache-dir


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 225.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 199.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 250.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 221.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 223.9 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.69-cp313-cp313-linux_x86_64.whl size=3624451 sha256=bc60d41ba184ba4d404b5c1ada8a78dd3e9ab6985829704e2ced99b740518052
  Stored in directory: /tmp/pip-ephem-wheel-cache-f1xf0fu7/wheels/8e/f9/ae/5414759be5654cb051c9db3d820747306b5ca2ddc05813c460
Successfully built llama-cpp-python
  Attempting uninstall: typing-extensions
    Found existing installation

# Loading the model

In [5]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-08-22 14:59:40--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 3.163.189.74, 3.163.189.37, 3.163.189.90, ...
Connecting to huggingface.co (huggingface.co)|3.163.189.74|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1787414380&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uP

In [16]:
!rm -f Phi-3-mini-4k-instruct-fp16.gguf

In [18]:
!wget https://huggingface.co

--2026-08-22 15:05:07--  https://huggingface.co/
Resolving huggingface.co (huggingface.co)... 3.163.189.74, 3.163.189.114, 3.163.189.37, ...
Connecting to huggingface.co (huggingface.co)|3.163.189.74|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 178145 (174K) [text/html]
Saving to: ‘index.html.2’

index.html.2        100%[===================>] 173.97K  --.-KB/s    in 0.01s   

2026-08-22 15:05:07 (12.2 MB/s) - ‘index.html.2’ saved [178145/178145]



In [27]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp

print("Downloading model from Hugging Face Hub (approx. 7.6GB)...")
# Safely download the precise quantized model file
model_local_path = hf_hub_download(
    repo_id="Microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="Phi-3-mini-4k-instruct-fp16.gguf"
)
print(f"Download complete! File saved to: {model_local_path}")

print("Loading model into GPU VRAM...")
# Pass the verified download path directly into LlamaCpp
llm = LlamaCpp(
    model_path=model_local_path,
    n_gpu_layers=-1,   # Offloads all layers to your Colab GPU
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

print("🚀 Success! Your Phi-3 model is fully loaded and ready to use.")


Download complete! File saved to: /root/.cache/huggingface/hub/models--Microsoft--Phi-3-mini-4k-instruct-gguf/snapshots/a64113399c2f6b8ad3e11c394733a2ddadaa7f33/Phi-3-mini-4k-instruct-fp16.gguf
Loading model into GPU VRAM...
🚀 Success! Your Phi-3 model is fully loaded and ready to use.


In [ ]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")